In [2]:
from pathlib import Path
import os

def _find_repo_root(start: Path) -> Path | None:
    for p in [start, *start.parents]:
        # Repo root has both package and tutorials folder
        if (p / "topobench").exists() and (p / "tutorials").exists():
            return p
    return None

# Prefer explicit workspace path, then fall back to cwd walk-up.
candidates = [
    Path("/Users/greciacastelazo/topobench"),
    Path.cwd(),
]

REPO_ROOT = None
for c in candidates:
    if c.exists() and (c / "topobench").exists() and (c / "tutorials").exists():
        REPO_ROOT = c
        break
    found = _find_repo_root(c)
    if found is not None:
        REPO_ROOT = found
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root. Set REPO_ROOT manually.")

os.chdir(REPO_ROOT)
print(f"Working directory set to: {REPO_ROOT}")

Working directory set to: /home/gcastelazo/TopoBench


In [3]:
# 1) Imports
import torch
import numpy as np
import lightning as pl
from omegaconf import OmegaConf

# Data loading / preprocessing utilities from the repo
from topobench.data.loaders.graph.a123_loader import A123DatasetLoader
from topobench.dataloader.dataloader import TBDataloader
from topobench.data.preprocessor import PreProcessor

# Model / training building blocks
from topobench.model.model import TBModel
# example backbone building block (SCN2 is optional; we provide a tiny custom backbone below)
# from topomodelx.nn.simplicial.scn2 import SCN2
from topobench.nn.wrappers.simplicial import SCNWrapper
from topobench.nn.encoders import AllCellFeatureEncoder
from topobench.nn.readouts import PropagateSignalDown

# Optimization / evaluation
from topobench.loss.loss import TBLoss
from topobench.optimizer import TBOptimizer
from topobench.evaluator.evaluator import TBEvaluator

print('Imports OK')

Imports OK


In [5]:
from pathlib import Path
import os

p = Path("/home/gcastelazo/TopoBench/data/a123_cortex_m/processed").resolve()
print("Path:", p)
print("Exists:", p.exists())
print("Writable dir:", os.access(p.parent, os.W_OK))
if p.exists():
    for f in p.glob("*"):
        print(f.name, "writable:", os.access(f, os.W_OK))

Path: /home/gcastelazo/TopoBench/data/a123_cortex_m/processed
Exists: False
Writable dir: False
